# 📊 Normalización de Datos — Sales Data Sample

Este notebook aplica técnicas de **normalización y preprocesamiento** al dataset de ventas.

## ¿Qué es la normalización?
La normalización reescala los valores numéricos a un rango estándar (ej: 0 a 1) para que:
- Los algoritmos de Machine Learning no favorezcan columnas con valores grandes.
- Las comparaciones entre variables sean justas.
- Los modelos converjan más rápido durante el entrenamiento.

## Técnicas que aplicaremos:
| Técnica | Rango resultante | Cuándo usarla |
|---|---|---|
| **Min-Max Scaling** | [0, 1] | Cuando los datos no tienen outliers extremos |
| **Z-Score (Standardization)** | Media=0, Std=1 | Cuando los datos siguen distribución normal |
| **Robust Scaler** | Basado en IQR | Cuando hay outliers que no quieres eliminar |

## 1. Instalación e importación de librerías

In [ ]:
# ── Librerías estándar de ciencia de datos ──────────────────────────────────
import pandas as pd          # Manejo de DataFrames (tablas de datos)
import numpy as np           # Operaciones matemáticas y arrays
import matplotlib.pyplot as plt   # Gráficos básicos
import seaborn as sns        # Gráficos estadísticos más bonitos

# ── Herramientas de normalización de scikit-learn ───────────────────────────
# scikit-learn es la librería estándar de Machine Learning en Python
from sklearn.preprocessing import (
    MinMaxScaler,      # Escala valores al rango [0, 1]
    StandardScaler,    # Estandariza a media=0 y desviación estándar=1 (Z-Score)
    RobustScaler,      # Escala usando la mediana e IQR — resistente a outliers
    LabelEncoder,      # Convierte categorías de texto a números enteros
    OneHotEncoder      # Convierte categorías a columnas binarias (0 o 1)
)

# ── Configuración visual ─────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')  # Estilo limpio para los gráficos
sns.set_palette('Set2')                  # Paleta de colores agradable
pd.set_option('display.max_columns', 30) # Mostrar todas las columnas del DataFrame
pd.set_option('display.float_format', '{:.4f}'.format)  # 4 decimales en floats

print('✅ Librerías cargadas correctamente')

## 2. Carga del dataset

> **Nota para Colab:** Sube el archivo `sales_data_sample.csv` usando el ícono de carpeta 📁 en el panel izquierdo, o ejecuta la celda siguiente para cargarlo desde tu computadora.

In [ ]:
# ── OPCIÓN A: Subir el archivo manualmente en Colab ──────────────────────────
from google.colab import files
uploaded = files.upload()   # Abre un diálogo para seleccionar el archivo

# ── OPCIÓN B: Cargar desde ruta local (si ya está subido) ────────────────────
FILE_PATH = 'sales_data_sample.csv'   # Cambiá esta ruta si tu archivo está en otra carpeta

# encoding='latin1' es necesario porque el archivo contiene caracteres especiales
# que no están en UTF-8 (el estándar). Latin1 es común en archivos exportados
# desde Excel en sistemas Windows o latinoamericanos.
df_original = pd.read_csv(FILE_PATH, encoding='latin1')

# Mostramos las primeras 5 filas para verificar que cargó bien
print(f'✅ Dataset cargado: {df_original.shape[0]} filas × {df_original.shape[1]} columnas')
df_original.head()

## 3. Exploración inicial del dataset (EDA)

In [ ]:
# ── Tipos de datos de cada columna ───────────────────────────────────────────
# Es importante saber si cada columna es numérica (int/float) o categórica (object)
# porque normalizaremos solo las numéricas y encodearemos las categóricas
print('📋 Tipos de datos por columna:')
print(df_original.dtypes)
print()

In [ ]:
# ── Estadísticas descriptivas de columnas numéricas ──────────────────────────
# count = cantidad de valores no nulos
# mean  = promedio
# std   = desviación estándar (cuánto se dispersan los valores alrededor del promedio)
# min/max = valores mínimos y máximos
# 25%/50%/75% = percentiles (el 50% es la mediana)
print('📊 Estadísticas descriptivas (columnas numéricas):')
df_original.describe()

In [ ]:
# ── Valores nulos por columna ────────────────────────────────────────────────
# Los valores nulos (NaN) son problemáticos para la normalización:
# los escaladores de sklearn fallan si hay NaN, hay que tratarlos antes.
print('🔍 Valores nulos por columna:')
nulos = df_original.isnull().sum()
nulos_pct = (nulos / len(df_original) * 100).round(2)

resumen_nulos = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': nulos_pct
})

# Mostramos solo las columnas que tienen al menos 1 nulo
print(resumen_nulos[resumen_nulos['Nulos'] > 0])

## 4. Preprocesamiento previo a la normalización

Antes de normalizar debemos:
1. Seleccionar las columnas relevantes
2. Tratar los valores nulos
3. Separar columnas numéricas de categóricas

In [ ]:
# ── Trabajamos sobre una copia del dataframe original ────────────────────────
# SIEMPRE trabajá sobre una copia para no perder los datos originales.
# Así podés comparar antes y después de la normalización.
df = df_original.copy()

# ── Columnas numéricas con significado analítico ─────────────────────────────
# Excluimos ORDERNUMBER y ORDERLINENUMBER porque son identificadores (IDs),
# no métricas reales. Normalizar un ID no tiene sentido.
COLS_NUMERICAS = [
    'QUANTITYORDERED',   # Cantidad de unidades pedidas por orden
    'PRICEEACH',         # Precio unitario del producto
    'SALES',             # Ventas totales de esa línea (cantidad × precio + descuentos)
    'MSRP',              # Precio de lista sugerido por el fabricante
    'QTR_ID',            # Trimestre del año (1-4)
    'MONTH_ID',          # Mes del año (1-12)
    'YEAR_ID'            # Año de la orden
]

# ── Columnas categóricas (texto) que codificaremos ───────────────────────────
COLS_CATEGORICAS = [
    'STATUS',       # Estado del pedido: Shipped, Cancelled, On Hold, etc.
    'PRODUCTLINE',  # Línea de producto: Classic Cars, Motorcycles, etc.
    'DEALSIZE'      # Tamaño del negocio: Small, Medium, Large
]

print('✅ Columnas definidas:')
print(f'  Numéricas ({len(COLS_NUMERICAS)}): {COLS_NUMERICAS}')
print(f'  Categóricas ({len(COLS_CATEGORICAS)}): {COLS_CATEGORICAS}')

In [ ]:
# ── Tratamiento de valores nulos ─────────────────────────────────────────────
# Las columnas numéricas que seleccionamos no tienen nulos (verificado en paso 3).
# Pero es buena práctica igual verificarlo y manejar el caso.

print('Nulos en columnas numéricas seleccionadas:')
print(df[COLS_NUMERICAS].isnull().sum())
print()

# Si hubiera nulos en columnas numéricas, podríamos:
# a) Rellenar con la media:   df[col].fillna(df[col].mean())
# b) Rellenar con la mediana: df[col].fillna(df[col].median())
# c) Eliminar las filas:      df.dropna(subset=col)

# Como no hay nulos en las columnas seleccionadas, continuamos.
print('✅ No hay valores nulos en las columnas numéricas seleccionadas. Listo para normalizar.')

## 5. Visualización ANTES de normalizar

Visualizamos las distribuciones originales para comparar con el resultado después.

In [ ]:
# ── Histogramas de distribución original ─────────────────────────────────────
# Un histograma muestra cuántas veces aparece cada rango de valores.
# Nos permite ver si los datos tienen distribución normal (campana) u otra forma.

fig, axes = plt.subplots(3, 3, figsize=(15, 10))  # 3 filas × 3 columnas de gráficos
axes = axes.flatten()   # Convertir matriz 3×3 a lista para iterar fácilmente

for i, col in enumerate(COLS_NUMERICAS):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'{col}\n(min={df[col].min():.1f}, max={df[col].max():.1f})', fontsize=10)
    axes[i].set_xlabel('Valor original')
    axes[i].set_ylabel('Frecuencia')

# Ocultamos el último subplot si quedó vacío (tenemos 7 cols en grilla de 9)
for j in range(len(COLS_NUMERICAS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('📊 Distribuciones ANTES de normalizar', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Nota: Observá la escala del eje X — cada variable tiene rangos muy diferentes.')
print('Eso es exactamente el problema que resuelve la normalización.')

## 6. Técnica 1 — Min-Max Scaling (Normalización [0, 1])

**Fórmula:** `x_norm = (x - x_min) / (x_max - x_min)`

- Transforma todos los valores al rango **[0, 1]**
- El valor mínimo pasa a ser **0**, el máximo a ser **1**
- **Pros:** Simple, intuitivo, mantiene la distribución original
- **Contras:** Muy sensible a outliers (un valor extremo distorsiona todo el rango)

In [ ]:
# ── Aplicar Min-Max Scaler ───────────────────────────────────────────────────

# 1. Instanciar el escalador
minmax_scaler = MinMaxScaler()   # Por defecto escala a rango (0, 1)
# Podés cambiar el rango: MinMaxScaler(feature_range=(-1, 1)) para (-1, 1)

# 2. fit_transform: aprende los min/max de cada columna Y transforma los datos
#    Equivale a hacer: scaler.fit(X) seguido de scaler.transform(X)
#    - fit()       → calcula min y max de cada columna
#    - transform() → aplica la fórmula de normalización
minmax_scaled = minmax_scaler.fit_transform(df[COLS_NUMERICAS])

# 3. El resultado es un numpy array; lo convertimos a DataFrame para trabajar mejor
df_minmax = pd.DataFrame(
    minmax_scaled,
    columns=[f'{col}_minmax' for col in COLS_NUMERICAS]  # Nombramos con sufijo _minmax
)

print('✅ Min-Max Scaling aplicado')
print('\nEstadísticas después de Min-Max Scaling:')
print(f'  Todos los mínimos → {df_minmax.min().min():.4f} (debería ser 0.0)')
print(f'  Todos los máximos → {df_minmax.max().max():.4f} (debería ser 1.0)')
print()
df_minmax.describe().round(4)

## 7. Técnica 2 — Z-Score / StandardScaler

**Fórmula:** `z = (x - μ) / σ`  donde μ=media y σ=desviación estándar

- Transforma los datos para que tengan **media = 0** y **desviación estándar = 1**
- No acota los valores a un rango fijo — pueden ser negativos o mayores a 1
- **Pros:** Funciona muy bien con algoritmos que asumen distribución normal (regresión, SVM)
- **Contras:** No acota el rango, menos intuitivo de interpretar

In [ ]:
# ── Aplicar StandardScaler (Z-Score) ────────────────────────────────────────

standard_scaler = StandardScaler()

# fit_transform: calcula la media y desviación estándar de cada columna
# y luego aplica la transformación Z = (X - media) / std
standard_scaled = standard_scaler.fit_transform(df[COLS_NUMERICAS])

df_standard = pd.DataFrame(
    standard_scaled,
    columns=[f'{col}_zscore' for col in COLS_NUMERICAS]
)

print('✅ Z-Score (StandardScaler) aplicado')
print('\nEstadísticas después de Z-Score:')
print(f'  Medias (deben ser ~0): {df_standard.mean().round(6).tolist()}')
print(f'  Std devs (deben ser ~1): {df_standard.std().round(4).tolist()}')
print()
df_standard.describe().round(4)

## 8. Técnica 3 — Robust Scaler

**Fórmula:** `x_rob = (x - mediana) / IQR`  donde IQR = Q75 - Q25

- Usa la **mediana** en vez del promedio y el **rango intercuartílico (IQR)** en vez del std
- Mucho más robusto ante **outliers** (valores extremos) que los dos anteriores
- **Pros:** Ideal cuando el dataset tiene outliers que no querés eliminar
- **Contras:** No garantiza rango fijo [0,1], valores extremos aún existen pero no distorsionan

In [ ]:
# ── Aplicar RobustScaler ─────────────────────────────────────────────────────

robust_scaler = RobustScaler()

# fit_transform: calcula la mediana e IQR de cada columna
# y aplica: (X - mediana) / IQR
robust_scaled = robust_scaler.fit_transform(df[COLS_NUMERICAS])

df_robust = pd.DataFrame(
    robust_scaled,
    columns=[f'{col}_robust' for col in COLS_NUMERICAS]
)

print('✅ RobustScaler aplicado')
print('\nEstadísticas después de Robust Scaling:')
df_robust.describe().round(4)

## 9. Encoding de variables categóricas

Los algoritmos de ML solo aceptan números. Las columnas de texto deben convertirse.

- **Label Encoding:** Asigna un número entero a cada categoría (ej: Small=0, Medium=1, Large=2). Útil cuando hay orden implícito.
- **One-Hot Encoding:** Crea una columna binaria (0/1) por cada categoría. Mejor cuando NO hay orden entre categorías.

In [ ]:
# ── Label Encoding ───────────────────────────────────────────────────────────
# Útil para DEALSIZE porque tiene un orden natural: Small < Medium < Large

label_encoder = LabelEncoder()
df_encoded = df.copy()

print('🔤 Label Encoding (asigna un número entero a cada categoría):')
for col in COLS_CATEGORICAS:
    df_encoded[f'{col}_label'] = label_encoder.fit_transform(df[col])
    # Mostramos el mapeo: qué número le corresponde a cada categoría
    mapeo = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
    print(f'  {col}: {mapeo}')

In [ ]:
# ── One-Hot Encoding ─────────────────────────────────────────────────────────
# Mejor opción para STATUS y PRODUCTLINE, donde NO hay un orden inherente.
# Crea columnas separadas tipo: STATUS_Shipped, STATUS_Cancelled, etc.

# pd.get_dummies es la forma más sencilla de hacer One-Hot Encoding en pandas
# drop_first=True elimina la primera categoría de cada variable para evitar
# multicolinealidad (el problema de variables linealmente dependientes)
df_ohe = pd.get_dummies(
    df[COLS_CATEGORICAS],
    drop_first=False,   # Ponemos False para ver todas las categorías en este ejemplo
    dtype=int           # Las columnas resultantes serán 0 o 1 (int, no bool)
)

print(f'✅ One-Hot Encoding aplicado')
print(f'   Columnas originales: {len(COLS_CATEGORICAS)}')
print(f'   Columnas resultantes: {len(df_ohe.columns)}')
print(f'   Nombres: {list(df_ohe.columns)}')
print()
df_ohe.head()

## 10. Dataset final consolidado

Combinamos todas las transformaciones en un único DataFrame listo para ML.

In [ ]:
# ── Construcción del DataFrame final ─────────────────────────────────────────
# Elegimos Min-Max como técnica principal para las numéricas
# (podés cambiar a df_standard o df_robust según tu caso de uso)

# Reseteamos los índices por seguridad antes de concatenar
df_final = pd.concat([
    df_minmax.reset_index(drop=True),   # Columnas numéricas normalizadas [0,1]
    df_ohe.reset_index(drop=True)       # Columnas categóricas en One-Hot
], axis=1)  # axis=1 = concatenar por columnas (no por filas)

print(f'✅ Dataset final construido')
print(f'   Dimensiones: {df_final.shape[0]} filas × {df_final.shape[1]} columnas')
print(f'   Todas las columnas son numéricas: {all(df_final.dtypes != object)}')
print()
df_final.head()

## 11. Comparación visual: antes vs. después

In [ ]:
# ── Comparación lado a lado: distribución antes y después ────────────────────
# Seleccionamos 3 columnas para comparar (las más representativas)
COLS_COMPARAR = ['QUANTITYORDERED', 'PRICEEACH', 'SALES']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))

for i, col in enumerate(COLS_COMPARAR):
    # Columna 0: datos originales
    axes[i, 0].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i, 0].set_title(f'{col}\nORIGINAL', fontsize=10, fontweight='bold')
    axes[i, 0].set_xlabel(f'Rango: [{df[col].min():.0f}, {df[col].max():.0f}]')

    # Columna 1: Min-Max
    axes[i, 1].hist(df_minmax[f'{col}_minmax'], bins=30, color='mediumseagreen', edgecolor='white', alpha=0.8)
    axes[i, 1].set_title(f'{col}\nMIN-MAX [0,1]', fontsize=10, fontweight='bold')
    axes[i, 1].set_xlabel('Rango: [0.0, 1.0]')

    # Columna 2: Z-Score
    axes[i, 2].hist(df_standard[f'{col}_zscore'], bins=30, color='tomato', edgecolor='white', alpha=0.8)
    axes[i, 2].set_title(f'{col}\nZ-SCORE (media=0, std=1)', fontsize=10, fontweight='bold')
    axes[i, 2].set_xlabel(f'Media≈0, Std≈1')

plt.suptitle('📊 Comparación: Datos Originales vs. Normalizados', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Boxplots comparativos para detectar outliers ─────────────────────────────
# Un boxplot muestra: mediana (línea central), IQR (caja), y outliers (puntos).
# Compararemos los 3 métodos de normalización en una sola columna: SALES

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

axes[0].boxplot(df['SALES'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[0].set_title('SALES\nOriginal', fontweight='bold')

axes[1].boxplot(df_minmax['SALES_minmax'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='mediumseagreen', alpha=0.7))
axes[1].set_title('SALES\nMin-Max [0,1]', fontweight='bold')

axes[2].boxplot(df_standard['SALES_zscore'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='tomato', alpha=0.7))
axes[2].set_title('SALES\nZ-Score', fontweight='bold')

axes[3].boxplot(df_robust['SALES_robust'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='mediumpurple', alpha=0.7))
axes[3].set_title('SALES\nRobust Scaler', fontweight='bold')

plt.suptitle('📦 Comparación de Outliers con Diferentes Escaladores — SALES', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Observá que el Robust Scaler comprime mejor los outliers extremos.')

## 12. Resumen de estadísticas finales

In [ ]:
# ── Tabla resumen: comparación de medias y desviaciones estándar ─────────────
resumen = pd.DataFrame({
    'Media Original':    df[COLS_NUMERICAS].mean().round(2),
    'Std Original':      df[COLS_NUMERICAS].std().round(2),
    'Media MinMax':      df_minmax.mean().values.round(4),
    'Std MinMax':        df_minmax.std().values.round(4),
    'Media Z-Score':     df_standard.mean().values.round(4),
    'Std Z-Score':       df_standard.std().values.round(4),
    'Media Robust':      df_robust.mean().values.round(4),
    'Std Robust':        df_robust.std().values.round(4),
}, index=COLS_NUMERICAS)

print('📋 Resumen: estadísticas antes y después de normalizar')
resumen

## 13. Guardar el dataset normalizado

In [ ]:
# ── Exportar el dataset final normalizado a CSV ──────────────────────────────
OUTPUT_PATH = 'sales_data_normalizado.csv'

# index=False evita que pandas agregue una columna extra con el índice de fila
df_final.to_csv(OUTPUT_PATH, index=False)
print(f'✅ Dataset normalizado guardado en: {OUTPUT_PATH}')
print(f'   Dimensiones finales: {df_final.shape[0]} filas × {df_final.shape[1]} columnas')

# ── Descarga directa en Google Colab ────────────────────────────────────────
# Descomenta las siguientes líneas si querés descargar el archivo a tu PC
# from google.colab import files
# files.download(OUTPUT_PATH)

print('\n🎉 ¡Proceso de normalización completo!')
print('\n📌 Recordatorio — ¿cuándo usar cada técnica?')
print('  • Min-Max Scaler  → Datos sin outliers extremos, necesitás rango [0,1]')
print('  • Z-Score         → Datos aproximadamente normales, para regresión/SVM/PCA')
print('  • Robust Scaler   → Datos con outliers que no querés eliminar')